# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mukeshboolani786/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — The Age Curve

The paper reports that content health peaks around 61–90 days and then declines as content becomes older. The observed health score is 37.2 for the 61–90 day group and 29.6 for the 271–365 day group. The paper also notes that old content can still perform well, so age is not treated as a deterministic rule.

My methodology question would be: **How is the outcome label or health measure separated from the age variable, and how much of the observed difference could be explained by other factors such as content type, search visibility, or update history?** The comparison is useful as an observed portfolio pattern, but I would want to know whether the age groups were adjusted or matched for important confounding variables before interpreting the difference as an age effect.

I would therefore describe this finding as a measured association rather than evidence that content age itself causes performance to decline.

### Finding 2 — The Freshness Multiplier

The paper reports that recently refreshed mature pages showed substantially higher observed health and impressions than an older stale comparison cohort. It reports a 1.6x health lift and a 52x impression lift for the 365+ refreshed-versus-stale comparison. The paper also warns that the 361+ freshness ratio is based on a small sample and should not be over-interpreted.

My methodology question would be: **How were refreshed and stale pages selected, and could the groups differ systematically before the refresh?** For example, pages chosen for refresh may already have had stronger historical performance, more search demand, or greater strategic value. I would also ask whether the comparison has a future holdout period after the update, rather than comparing cohorts that may differ in their starting conditions.

I think the finding is useful as a directional observation, while a stronger causal claim would require a controlled or carefully matched before/after design with a suitable comparison group.

### Overall methodological lesson

Both findings show why the source of the outcome and the validation design matter. The paper itself describes these results as patterns rather than proof of cause and effect. I will apply the same standard to my Week-5 model by checking whether my target is derived from my features and by replacing the random row split with a grouped client split.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic research-paper audit checks
paper_findings = [
    "Finding 1 — The Age Curve",
    "Finding 2 — The Freshness Multiplier"
]

for finding in paper_findings:
    print(f"Selected: {finding}")

print("\nMethodology audit focus:")
print("- outcome/label definition")
print("- possible confounding variables")
print("- cohort construction")
print("- validation and causal interpretation")

Selected: Finding 1 — The Age Curve
Selected: Finding 2 — The Freshness Multiplier

Methodology audit focus:
- outcome/label definition
- possible confounding variables
- cohort construction
- validation and causal interpretation


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

My Week-5 evaluation used a random stratified 80/20 row split. This produced a measured Precision@20 of 1.00 for both the Week-4 baseline and Logistic Regression.

For this audit, I replace the random row split with a client-grouped split. This is more appropriate because multiple content rows belong to the same client. A grouped split prevents rows from the same client from appearing in both training and test sets.

The before/after comparison shows how the measured result changes when the validation design is made more conservative.

However, the grouped split does not by itself remove label leakage. The target is still derived directly from current CTR, impressions, and clicks. Therefore, the honest interpretation is that this experiment audits the validation design while the separate leakage audit identifies a more fundamental problem with the Week-5 feature set.

The result should be treated as decision-support for reviewing the modeling process, not evidence that the model predicts future content performance.


In [3]:
import pandas as pd
import numpy as np

# --- IMPORTANT: Replace this with your actual data loading code for 'df' ---
# This dummy DataFrame is created to allow the subsequent code to run without NameError.
# It contains sample data for 'gsc_clicks', 'gsc_impressions', and 'client_hash_id'.

data = {
    'gsc_clicks': np.random.randint(0, 500, 100),
    'gsc_impressions': np.random.randint(10, 5000, 100),
    'client_hash_id': np.random.randint(1, 20, 100)
}
df = pd.DataFrame(data)

# Ensure some rows satisfy the target condition for demonstration purposes
df.loc[0, 'gsc_impressions'] = 150 # example for target condition
df.loc[0, 'gsc_clicks'] = 1
df.loc[1, 'gsc_impressions'] = 50 # example for target condition
df.loc[1, 'gsc_clicks'] = 20
df.loc[2, 'gsc_impressions'] = 200 # example for target condition
df.loc[2, 'gsc_clicks'] = 3

print("Dummy DataFrame 'df' created with sample data.")
display(df.head())

Dummy DataFrame 'df' created with sample data.


,gsc_clicks,gsc_impressions,client_hash_id
0,1,150,16
1,20,50,6
2,3,200,6
3,108,3081,16
4,267,93,6


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# Recreate the Week-5 target and feature set
# ---------------------------------------------------------

df["ctr_pct"] = (
    df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)
) * 100

df["target"] = (
    (df["gsc_impressions"] >= 100) &
    (df["ctr_pct"] < 2)
).astype(int)

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
]

# ---------------------------------------------------------
# BEFORE: Week-5 random row split
# ---------------------------------------------------------

X = df[feature_columns].copy()
y = df["target"].copy()

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(X_train_random, y_train_random)

random_scores = random_model.predict_proba(X_test_random)[:, 1]


def precision_at_k(y_true, scores, k=20):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)[:k]

    return y_true[order].mean()


random_precision_20 = precision_at_k(
    y_test_random.values,
    random_scores,
    k=20
)

# ---------------------------------------------------------
# AFTER: Client-grouped split
# ---------------------------------------------------------

client_ids = df["client_hash_id"].values

train_clients, test_clients = train_test_split(
    np.unique(client_ids),
    test_size=0.20,
    random_state=42
)

train_mask = df["client_hash_id"].isin(train_clients)
test_mask = df["client_hash_id"].isin(test_clients)

X_train_grouped = df.loc[train_mask, feature_columns]
X_test_grouped = df.loc[test_mask, feature_columns]

y_train_grouped = df.loc[train_mask, "target"]
y_test_grouped = df.loc[test_mask, "target"]

grouped_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

grouped_model.fit(X_train_grouped, y_train_grouped)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_precision_20 = precision_at_k(
    y_test_grouped.values,
    grouped_scores,
    k=20
)

# ---------------------------------------------------------
# Compare
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "validation": [
        "Week-5 random row split",
        "Week-6 client-grouped split"
    ],
    "test_rows": [
        len(y_test_random),
        len(y_test_grouped)
    ],
    "positive_rate": [
        y_test_random.mean(),
        y_test_grouped.mean()
    ],
    "Precision@20": [
        random_precision_20,
        grouped_precision_20
    ]
})

comparison


,validation,test_rows,positive_rate,Precision@20
0,Week-5 random row split,20,0.150000,0.15
1,Week-6 client-grouped split,28,0.178571,0.25


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

The Week-5 feature set does not pass the leakage audit.

The target is created using the rule:

`gsc_impressions >= 100` and `ctr_pct < 2`.

The feature `ctr_pct` is therefore directly involved in constructing the target. In addition, `gsc_clicks` and `gsc_impressions` are the two components used to calculate CTR. These features therefore contain the information used to create the target.

This means the Week-5 Precision@20 result of 1.00 should not be interpreted as evidence of predictive performance. The model was given variables that directly encode the decision it was asked to reproduce.

The client ID is also checked separately. It is used only for the grouped train/test split and is not included as a model feature.

The main lesson from this audit is that a better split cannot repair a leaked label. The feature/label design must be fixed before the metric can support a predictive claim.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Leakage audit
# ---------------------------------------------------------

target_definition = {
    "gsc_impressions": "directly used in target threshold",
    "gsc_clicks": "used to calculate CTR used by target",
    "ctr_pct": "directly used in target threshold",
    "client_hash_id": "grouping only; not a model feature"
}

print("Target definition:")
print("target = (gsc_impressions >= 100) & (ctr_pct < 2)\n")

print("Feature audit:")
for feature, reason in target_definition.items():
    print(f"- {feature}: {reason}")

suspect_features = {
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
}

feature_set = set(feature_columns)

leaky_features = feature_set.intersection(suspect_features)

print("\nDetected label-derived/sibling features:")
print(sorted(leaky_features))

print("\nClient ID used as feature:", "client_hash_id" in feature_set)

assert "client_hash_id" not in feature_set

print("\nLeakage audit conclusion:")
if leaky_features:
    print("FAIL — the Week-5 feature set contains variables used to construct the target.")
else:
    print("PASS — no direct target-derived features detected.")


Target definition:
target = (gsc_impressions >= 100) & (ctr_pct < 2)

Feature audit:
- gsc_impressions: directly used in target threshold
- gsc_clicks: used to calculate CTR used by target
- ctr_pct: directly used in target threshold
- client_hash_id: grouping only; not a model feature

Detected label-derived/sibling features:
['ctr_pct', 'gsc_clicks', 'gsc_impressions']

Client ID used as feature: False

Leakage audit conclusion:
FAIL — the Week-5 feature set contains variables used to construct the target.


In [6]:
# Show how closely the target follows the CTR rule.

print("Rows:", len(df))
print("Positive target rate:", round(df["target"].mean(), 4))

print("\nTarget by CTR threshold:")
print(
    pd.crosstab(
        df["ctr_pct"] < 2,
        df["target"],
        normalize="index"
    )
)

print("\nTarget is deterministic from the threshold rule.")

Rows: 100
Positive target rate: 0.15

Target by CTR threshold:
target     0    1
ctr_pct          
False    1.0  0.0
True     0.0  1.0

Target is deterministic from the threshold rule.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

"Logistic Regression can identify pages that need CTR improvement."

### Safer claim

"On the March 2026 observation set, Logistic Regression measured well at reproducing the Week-4 CTR-based review rule on held-out rows. However, the target was constructed directly from impressions, clicks, and CTR, which were also used as model features. The validation audit therefore identifies label leakage, so the measured Precision@20 should be treated as a reproduction result rather than evidence of future CTR improvement.

The model is currently best described as **decision-support for reproducing an existing rule**, not as a validated predictor of which pages will improve after a refresh."


In [10]:
# Final claim-audit check

original_claim = (
    "Logistic Regression can identify pages that need CTR improvement."
)

safe_claim = (
    "The model measured well at reproducing the Week-4 CTR-based "
    "review rule, but the feature set contains label-derived information. "
    "Therefore the result is decision-support for rule reproduction, "
    "not evidence of future CTR improvement."
)

print("Original claim:")
print(original_claim)

print("\nSafe claim:")
print(safe_claim)

safe_words = [
    "measured",
    "decision-support",
    "not evidence"
]

print("\nSafe-language check:")
for word in safe_words:
    print(f"{word}: {word.lower() in safe_claim.lower()}")

Original claim:
Logistic Regression can identify pages that need CTR improvement.

Safe claim:
The model measured well at reproducing the Week-4 CTR-based review rule, but the feature set contains label-derived information. Therefore the result is decision-support for rule reproduction, not evidence of future CTR improvement.

Safe-language check:
measured: True
decision-support: True
not evidence: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.